importing necessary libraries and the dataset

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "iframe"
import plotly.express as px
from textblob import TextBlob
import plotly.graph_objects as go

In [ ]:
df = pd.read_csv("/Users/abhimanyuchettiar/Downloads/swiggy.csv")

Data cleaning

In [ ]:
df.head()

In [ ]:
# Create a copy for cleaning
df_clean = df.copy()

# A. Convert Price to numeric (removing any ₹ or commas if present)
df_clean['Price'] = pd.to_numeric(df_clean['Price'].astype(str).str.replace('₹', '').str.replace(',', ''), errors='coerce')

# B. Convert Avg ratings to numeric (handles 'NEW' or '--' cases)
df_clean['Avg ratings'] = pd.to_numeric(df_clean['Avg ratings'], errors='coerce')

# C. Handle Missing Values
# Fill missing ratings with the median so we don't lose too much data
df_clean['Avg ratings'] = df_clean['Avg ratings'].fillna(df_clean['Avg ratings'].median())

# D. Drop rows with no Price or Restaurant name
df_clean.dropna(subset=['Price', 'Restaurant'], inplace=True)

print(f"Dataset cleaned. Rows remaining: {len(df_clean)}")

Data Analysis

Top 10 cities by restaurant count

In [ ]:
city_counts = df_clean['City'].value_counts().head(10).reset_index()
city_counts.columns = ['City', 'Count']

fig1 = px.bar(city_counts, x='City', y='Count', 
             title='Top 10 Cities with Most Restaurants',
             color='Count', color_continuous_scale='Magma_r')
fig1.show()

Price distribution (cost for 2)

In [ ]:
fig2 = px.histogram(df_clean, x='Price', nbins=40, 
                   title='Distribution of Restaurant Prices',
                   labels={'Price': 'Price (₹)'},
                   marginal='violin', # Adds a violin plot to show density
                   color_discrete_sequence=['#636EFA'])
fig2.show()

Price vs avg rating correlation

In [ ]:
# Using a sample of 3000 to keep the Plotly graph smooth
sample_df = df_clean.sample(min(3000, len(df_clean)))

fig3 = px.scatter(sample_df, x='Price', y='Avg ratings', 
                 color='Avg ratings', size='Total ratings',
                 hover_name='Restaurant', opacity=0.5,
                 title='Correlation: Price vs. Rating',
                 labels={'Price': 'Price (₹)', 'Avg ratings': 'Rating'})
fig3.show()

Top 10 popular cuisines

In [ ]:
# Split the "Food type" string and count individual cuisines
cuisine_split = df_clean['Food type'].str.split(',').explode().str.strip()
top_cuisines = cuisine_split.value_counts().head(10).reset_index()
top_cuisines.columns = ['Food Type', 'Count']

fig4 = px.pie(top_cuisines, values='Count', names='Food Type', 
             title='Top 10 Food Types in the Dataset',
             hole=0.4)
fig4.show()

Delivery time vs ratings

In [ ]:
# Grouping by rating to see average delivery time
delivery_trend = df_clean.groupby('Avg ratings')['Delivery time'].mean().reset_index()

fig5 = px.line(delivery_trend, x='Avg ratings', y='Delivery time', 
              title='Trend: Average Delivery Time vs. Customer Rating',
              markers=True, line_shape='vh') # Stepped line for clarity
fig5.show()

Rating density by city

In [ ]:
top_5_cities = df_clean['City'].value_counts().head(5).index
city_box_df = df_clean[df_clean['City'].isin(top_5_cities)]

fig6 = px.box(city_box_df, x='City', y='Avg ratings', 
             color='City', title='Rating Distribution Across Top 5 Cities',
             points="all") # Shows all individual restaurant dots next to box
fig6.show()

feature engineering

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import numpy as np

# A. Feature Selection
# We'll use Price, City, and Delivery time to predict Avg ratings
features = ['Price', 'City', 'Delivery time']
X = df_clean[features]
y = df_clean['Avg ratings']

# B. Categorical Encoding
# Linear models can't read "Bangalore", so we turn cities into binary columns
X = pd.get_dummies(X, columns=['City'], drop_first=True)

# C. Train-Test Split
# We keep 20% of data aside to "test" how the model performs on new data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
from sklearn.linear_model import LinearRegression

# Initialize and train
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test)

# Evaluate
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression -> MAE: {mae_lr:.2f}, R2 Score: {r2_lr:.2f}")

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Initialize and train
dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

# Predict
y_pred_dt = dt_model.predict(X_test)

# Evaluate
mae_dt = mean_absolute_error(y_test, y_pred_dt)
print(f"Decision Tree -> MAE: {mae_dt:.2f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Initialize and train
rf_model = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test)

# Evaluate
mae_rf = mean_absolute_error(y_test, y_pred_rf)
print(f"Random Forest -> MAE: {mae_rf:.2f}")

In [ ]:
import plotly.graph_objects as go

# Create a comparison dataframe
comparison_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_rf}).sample(200)

fig_ml = go.Figure()

# Plot Actual values
fig_ml.add_trace(go.Scatter(y=comparison_df['Actual'], mode='markers', name='Actual Rating', marker=dict(color='blue', opacity=0.5)))

# Plot Predicted values
fig_ml.add_trace(go.Scatter(y=comparison_df['Predicted'], mode='markers', name='Predicted Rating', marker=dict(color='red', symbol='x')))

fig_ml.update_layout(title='Random Forest: Actual vs. Predicted Ratings (Sample of 200)',
                   xaxis_title='Restaurant Index',
                   yaxis_title='Rating',
                   template='plotly_white')
fig_ml.show()

In [ ]:
# Get importance from the Random Forest model
importances = rf_model.feature_importances_
feature_names = X.columns
feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False).head(10)

fig_imp = px.bar(feat_imp_df, x='Importance', y='Feature', orientation='h',
                 title='Top Factors Influencing Restaurant Ratings',
                 color='Importance', color_continuous_scale='Blues')
fig_imp.update_layout(yaxis={'categoryorder':'total ascending'})
fig_imp.show()